<a href="https://colab.research.google.com/github/volsarino/-/blob/main/%E6%89%8B%E8%BF%BD%E5%BE%93%E3%83%A2%E3%83%87%E3%83%AB(%E8%87%AA%E4%BD%9CViT%E3%83%A2%E3%83%87%E3%83%AB%E4%BD%BF%E7%94%A8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
BATCH_SIZE = 64
IMAGE_SIZE = 224
PATCH_SIZE = 16
NUM_CLASSES = 10

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

#データセットのダウンロード
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE,
                                          shuffle=True, num_workers=2)

#データの形状確認
data_iter = iter(trainloader)
images, labels = next(data_iter)
print(f"Input Shape: {images.shape}")

100%|██████████| 170M/170M [47:49<00:00, 59.4kB/s]


Input Shape: torch.Size([64, 3, 224, 224])


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, emb_size=768, img_size=224):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.projection = nn.Sequential(
            nn.Conv2d(in_channels, emb_size, kernel_size=patch_size, stride=patch_size),
            Rearrange('b e h w -> b (h w) e')
        )

    def forward(self, x):
        x = self.projection(x)
        return x

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, emb_size=768, img_size=224):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        self.projection = nn.Sequential(
            nn.Conv2d(in_channels, emb_size, kernel_size=patch_size, stride=patch_size),
            Rearrange('b e h w -> b (h w) e')
        )

        #CLSトークン
        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_size))
        self.positions = nn.Parameter(torch.randn(1, self.n_patches + 1, emb_size))

    def forward(self, x):
        b, _, _, _ = x.shape
        x = self.projection(x)
        #バッチサイズ分だけCLSトークンを複製
        cls_tokens = repeat(self.cls_token, '() n e -> b n e', b=b)
        x = torch.cat([cls_tokens, x], dim=1)
        # 位置情報を加算
        x += self.positions
        return x

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size=768, num_heads=12, dropout=0.):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.head_dim = emb_size // num_heads
        self.qkv = nn.Linear(emb_size, emb_size * 3)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x):
        qkv = rearrange(self.qkv(x), 'b n (h d qkv) -> (qkv) b h n d', h=self.num_heads, qkv=3)
        queries, keys, values = qkv[0], qkv[1], qkv[2]
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)
        scaling = self.emb_size ** (1/2)
        att = torch.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav', att, values)
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.projection(out)

        return out

In [ ]:
class FeedForward(nn.Sequential):
    def __init__(self, emb_size, expansion=4, drop_p=0.):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
            nn.Dropout(drop_p)
        )

class TransformerEncoderBlock(nn.Module):
    def __init__(self, emb_size=768, num_heads=12, forward_expansion=4, drop_p=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(emb_size)
        self.mha = MultiHeadAttention(emb_size, num_heads, drop_p)
        self.norm2 = nn.LayerNorm(emb_size)
        self.ff = FeedForward(emb_size, forward_expansion, drop_p)

        self.dropout = nn.Dropout(drop_p)

    def forward(self, x):
        x = x + self.dropout(self.mha(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

In [ ]:
class TransformerEncoder(nn.Sequential):
    def __init__(self, depth=12, **kwargs):
        super().__init__(*[TransformerEncoderBlock(kwargs) for _ in range(depth)])

In [ ]:
class ViT(nn.Module):
    def __init__(self,
                in_channels=3,
                patch_size=16,
                emb_size=768,
                img_size=224,
                depth=12,
                num_classes=10,
                **kwargs):
        super().__init__()
        self.patch_embedding = PatchEmbedding(in_channels, patch_size, emb_size, img_size)
        self.transformer_encoder = TransformerEncoder(depth, emb_size=emb_size, **kwargs)
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(emb_size),
            nn.Linear(emb_size, num_classes)
        )

    def forward(self, x):
        x = self.patch_embedding(x)
        x = self.transformer_encoder(x)
        x = self.mlp_head(x[:, 0])
        return x

In [ ]:
# モデルのインスタンス化
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViT(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

model.train()
for i, (inputs, labels) in enumerate(trainloader):
    inputs, labels = inputs.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    if i % 100 == 0:
        print(f'Batch {i}, Loss: {loss.item():.4f}')

TypeError: empty() received an invalid combination of arguments - got (tuple, dtype=NoneType, device=NoneType), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.memory_format memory_format = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, torch.memory_format memory_format = None, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [ ]:
def visualize_attention(image, attention_weights):
    cls_attention = attention_weights[0, 0, 1:].detach().cpu().numpy()

    attention_map = cls_attention.reshape(14, 14)
    attention_map = cv2.resize(attention_map, (224, 224))

    attention_map = (attention_map - attention_map.min()) / (attention_map.max() - attention_map.min())

    # 可視化のセットアップ
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    # 元画像の表示
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Attention Mapをヒートマップとして重ねて表示
    axes[1].imshow(image)
    axes[1].imshow(attention_map, cmap="jet", alpha=0.5)
    axes[1].set_title("Attention Map")
    axes[1].axis("off")

    plt.show()